# Content-Based Filtering

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import joblib
import os

## Load Data & Build TF-IDF Matrix

In [2]:
movies = pd.read_csv('../data/movies_clean.csv')

if 'genres_clean' not in movies.columns:
    movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Build TF-IDF matrix on genre tokens
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print(f'Number of movies : {len(movies):,}')
print(f'TF-IDF matrix : {tfidf_matrix.shape}')
print(f'Unique genres : {len(tfidf.vocabulary_)}')
print(f'Genre vocabulary : {sorted(tfidf.vocabulary_.keys())}')

Number of movies : 9,742
TF-IDF matrix : (9742, 22)
Unique genres : 22
Genre vocabulary : ['action', 'adventure', 'animation', 'children', 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'fi', 'film', 'horror', 'imax', 'musical', 'mystery', 'noir', 'romance', 'sci', 'thriller', 'unknown', 'war', 'western']


## Compute Cosine Similarity

In [3]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print(f'Cosine similarity matrix shape: {cosine_sim.shape}')
print(f'Values range from 0.0 (no overlap) to 1.0 (identical genres)')

Cosine similarity matrix shape: (9742, 9742)
Values range from 0.0 (no overlap) to 1.0 (identical genres)


## Recommendation Function

In [4]:
def content_recommend(title, top_n=10):
    if title not in indices:
        return f"Movie '{title}' not found in dataset."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort from highest to lowest; skip index 0 (the seed movie itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1 : top_n + 1]

    movie_indices   = [i[0] for i in sim_scores]
    similarity_vals = [round(i[1], 4) for i in sim_scores]

    result = movies[['title', 'genres']].iloc[movie_indices].copy()
    result['similarity_score'] = similarity_vals
    result.reset_index(drop=True, inplace=True)
    return result

def recommendat (recommendation):
    if isinstance(recommendation, str):
        print(recommendation)

    else:
        for i in range(len(recommendation)):
            print(
                f"{i+1}. "
                f"{recommendation.iloc[i]['title']} | "
                f"{recommendation.iloc[i]['genres']} | "
                f"Score: {recommendation.iloc[i]['similarity_score']}"
            )

print('Recommendations for Toy Story (1995) :')
recommendat(content_recommend('Toy Story (1995)'))

print('Recommendations for Dark Knight, The (2008) :')
recommendat(content_recommend('Dark Knight, The (2008)'))

x = "Iron Man (2008)"
print(f'Recommendations for "{x}":')
recommend = content_recommend(x)

for i in range(len(recommend)):
    print(
        f"{i+1}. "
        f"{recommend.iloc[i]['title']} | "
        f"{recommend.iloc[i]['genres']} | "
        f"Score: {recommend.iloc[i]['similarity_score']}")

Recommendations for Toy Story (1995) :
1. Antz (1998) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
2. Toy Story 2 (1999) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
3. Adventures of Rocky and Bullwinkle, The (2000) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
4. Emperor's New Groove, The (2000) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
5. Monsters, Inc. (2001) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
6. Wild, The (2006) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
7. Shrek the Third (2007) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
8. Tale of Despereaux, The (2008) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
9. Asterix and the Vikings (Astérix et les Vikings) (2006) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
10. Turbo (2013) | Adventure|Animation|Children|Comedy|Fantasy | Score: 1.0
Recommendations for Dark Knight, The (2008) :
1. Need for Speed (20

## Evaluation: Precision@K, Recall@K, F1@K

In [5]:
def get_relevant_movies(title):
    idx = indices[title]
    seed_genres = set(movies.loc[idx, 'genres'].split('|'))

    relevant = set()
    for _, row in movies.iterrows():
        if row['title'] == title:
            continue
        movie_genres = set(row['genres'].split('|'))
        if seed_genres & movie_genres:
            relevant.add(row['title'])
    return relevant


def evaluate_content_based(sample_size=200, top_k=10, random_state=42):
    np.random.seed(random_state)
    sample_titles = np.random.choice(movies['title'].values, size=sample_size, replace=False)

    precisions, recalls, f1s = [], [], []

    for title in sample_titles:
        if title not in indices:
            continue

        relevant = get_relevant_movies(title)
        if not relevant:
            continue

        recs = content_recommend(title, top_n=top_k)
        if isinstance(recs, str):
            continue

        recommended = set(recs['title'].tolist())
        hits        = len(recommended & relevant)

        precision = hits / top_k
        recall = hits / min(len(relevant), top_k) 
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    print(f'Content-Based Filtering — Evaluation (sample={sample_size}, K={top_k})')
    print(f'Relevance criterion  : at least one shared genre')
    print(f'Precision@{top_k} : {np.mean(precisions):.4f}')
    print(f'Recall@{top_k} : {np.mean(recalls):.4f}')
    print(f'F1@{top_k} : {np.mean(f1s):.4f}')
    return np.mean(precisions), np.mean(recalls), np.mean(f1s)
p, r, f1 = evaluate_content_based()

Content-Based Filtering — Evaluation (sample=200, K=10)
Relevance criterion  : at least one shared genre
Precision@10 : 0.9680
Recall@10 : 0.9680
F1@10 : 0.9680


## Save Models


In [6]:
os.makedirs('../models', exist_ok=True)

joblib.dump(tfidf,      '../models/tfidf_vectorizer.pkl')
joblib.dump(cosine_sim, '../models/cosine_similarity.pkl')
joblib.dump(indices,    '../models/movie_indices.pkl')
print('Models saved')

Models saved
